# 09b — International panel: local carry/value controls

This notebook is a single pre-submission diagnostic requested after the canonical panel analysis. It does **not** retune the predictor. It asks whether the country-and-month fixed-effects slope of the past-only macro-anchor signal survives controls for two simple local bond characteristics: the ten-year yield level and the ten-year-minus-three-month slope. Both controls are standardized strictly from each country's past history using the same 120-month burn-in as the predictor. Driscoll–Kraay inference uses 12 lags.

The frozen input is `g10_panel_model_objects.csv` from notebook 09 v7, SHA-256 `1628c4eaff487f6d87cc93fe62c7af909850d9ceedf448d9d8321f8b8477bbf6`.

In [1]:
from pathlib import Path
import hashlib, numpy as np, pandas as pd

INPUT = Path("g10_panel_model_objects.csv")
EXPECTED_SHA = "1628c4eaff487f6d87cc93fe62c7af909850d9ceedf448d9d8321f8b8477bbf6"
MINW = 120
NW_LAG = 12

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

if not INPUT.exists():
    # Colab-friendly recursive search
    hits=list(Path('.').rglob(INPUT.name))
    if not hits: raise FileNotFoundError(INPUT.name)
    INPUT=hits[0]
sha=sha256_file(INPUT)
assert sha==EXPECTED_SHA, (sha, EXPECTED_SHA)
model_panel=pd.read_csv(INPUT,parse_dates=['date']).sort_values(['country','date']).reset_index(drop=True)
print('verified input:', INPUT, sha)
print('rows:', len(model_panel), 'countries:', model_panel.country.nunique())

verified input: g10_panel_model_objects.csv 1628c4eaff487f6d87cc93fe62c7af909850d9ceedf448d9d8321f8b8477bbf6
rows: 5850 countries: 10


In [2]:
def past_only_z(x, minobs=MINW):
    x=np.asarray(x,float); z=np.full(len(x),np.nan)
    for t in range(minobs,len(x)):
        hist=x[:t]; hist=hist[np.isfinite(hist)]
        if len(hist)<minobs or not np.isfinite(x[t]): continue
        sd=hist.std(ddof=1)
        if sd>1e-12: z[t]=(x[t]-hist.mean())/sd
    return z

for col in ['y10','slope']:
    model_panel[col+'_z_past']=model_panel.groupby('country',group_keys=False)[col].transform(
        lambda s: pd.Series(past_only_z(s.to_numpy()),index=s.index))
analysis=model_panel.dropna(subset=['rx','tp','tp_z_past']).copy()
print('analysis sample:',len(analysis),'obs,',analysis.date.nunique(),'months,',analysis.country.nunique(),'countries')

analysis sample: 3351 obs, 532 months, 9 countries


In [3]:
def alternating_absorb(df, cols, groups, tol=1e-11, max_iter=500):
    z=df[cols].astype(float).copy()
    for _ in range(max_iter):
        old=z.to_numpy(copy=True)
        for g in groups:
            z=z-z.groupby(df[g]).transform('mean')
        if np.nanmax(np.abs(z.to_numpy()-old))<tol: break
    return z

def driscoll_kraay(y,X,dates,lag=NW_LAG):
    y=np.asarray(y,float); X=np.asarray(X,float); dates=pd.to_datetime(np.asarray(dates))
    if X.ndim==1: X=X[:,None]
    m=np.isfinite(y)&np.isfinite(X).all(axis=1)&pd.notna(dates)
    y,X,dates=y[m],X[m],dates[m]
    beta=np.linalg.lstsq(X,y,rcond=None)[0]
    e=y-X@beta; bread=np.linalg.pinv(X.T@X)
    score=pd.DataFrame(X*e[:,None]); score['date']=dates
    G=score.groupby('date').sum().sort_index().to_numpy(float)
    S=G.T@G
    for ell in range(1,min(lag,len(G)-1)+1):
        w=1-ell/(lag+1); Gamma=G[ell:].T@G[:-ell]; S+=w*(Gamma+Gamma.T)
    V=bread@S@bread; T=len(np.unique(dates)); k=X.shape[1]
    if T>k: V*=T/(T-k)
    se=np.sqrt(np.maximum(np.diag(V),0))
    return beta,se,int(len(y)),int(T)

def panel_fit(data,x_cols):
    d=data[['date','country','rx']+x_cols].dropna().copy()
    z=alternating_absorb(d,['rx']+x_cols,['country','date'])
    b,se,n,T=driscoll_kraay(z['rx'].to_numpy(),z[x_cols].to_numpy(),d['date'])
    return pd.DataFrame({'beta':b,'se_DK':se,'t_DK':b/se},index=x_cols),n,T

models={
 'TP only':['tp_z_past'],
 'TP + local yield level':['tp_z_past','y10_z_past'],
 'TP + local slope (carry)':['tp_z_past','slope_z_past'],
 'TP + level + slope':['tp_z_past','y10_z_past','slope_z_past'],
}
rows=[]
for name,xs in models.items():
    fit,n,T=panel_fit(analysis,xs)
    for var,row in fit.iterrows():
        rows.append({'model':name,'variable':var,**row.to_dict(),'n':n,'T':T})
results=pd.DataFrame(rows)
results

,model,variable,beta,se_DK,t_DK,n,T
0,TP only,tp_z_past,0.018547,0.003836,4.835093,3351,532
1,TP + local yield level,tp_z_past,0.018827,0.003841,4.901590,3351,532
2,TP + local yield level,y10_z_past,0.019285,0.008831,2.183771,3351,532
3,TP + local slope (carry),tp_z_past,0.015877,0.004829,3.287734,3351,532
4,TP + local slope (carry),slope_z_past,0.004557,0.004284,1.063663,3351,532
5,TP + level + slope,tp_z_past,0.013428,0.004353,3.084908,3351,532
6,TP + level + slope,y10_z_past,0.024453,0.008544,2.862031,3351,532
7,TP + level + slope,slope_z_past,0.009341,0.004473,2.088153,3351,532


In [4]:
# Hard check: the baseline must reproduce notebook 09 v7.
base=results[(results.model=='TP only')&(results.variable=='tp_z_past')].iloc[0]
assert abs(base.beta-0.018547)<5e-6
assert abs(base.t_DK-4.8351)<5e-4

headline=results[results.variable=='tp_z_past'][['model','beta','se_DK','t_DK','n','T']].copy()
print(headline.to_string(index=False,float_format=lambda x:f'{x:.6f}'))
results.to_csv('table_panel_carry_value_controls.csv',index=False)
print('\nRESULT: TP slope remains positive and significant after local level and slope controls; joint-control beta = %.4f, DK t = %.2f.' % (headline.iloc[-1].beta, headline.iloc[-1].t_DK))

                   model     beta    se_DK     t_DK    n   T
                 TP only 0.018547 0.003836 4.835093 3351 532
  TP + local yield level 0.018827 0.003841 4.901590 3351 532
TP + local slope (carry) 0.015877 0.004829 3.287734 3351 532
      TP + level + slope 0.013428 0.004353 3.084908 3351 532

RESULT: TP slope remains positive and significant after local level and slope controls; joint-control beta = 0.0134, DK t = 3.08.
